### Imports

In [ ]:
import matplotlib.pyplot as plt
import pybamm

from aepybamm import get_params

### Define source parameters

We use the About:Energy Gen1 demo cell (fictitious cell for technology demonstration).

In [ ]:
fp_bpx = "params/Molicel_P45B_bpx.json"

### Call `get_params()` with a defined degradation state

Compare the vanilla approach in `molicel_p45b_example.ipynb`. Here, the `degradation_state` input to `get_params()` defines a thermodynamic degradation state. Because of the capacity fade implied by the degradation state, less charge is passed in each of the voltage-controlled constant-current steps.

*Note: The warning returned from the BPX validator is due to slight imprecision in the stoichiometry specification of the parameter set (to be fixed with updated process).*

In [ ]:
degradation_state = {
    "LAM_NE": 0.05,
    "LAM_PE": 0.025,
    "LLI": 0.1,
}

parameter_values, model = get_params(
    fp_bpx,
    SOC_init=0.9,
    degradation_state=degradation_state,
    blended_electrode= (True, False),  # neg, pos
)

experiment = pybamm.Experiment(
    [
        "Discharge at 1C until 3 V",
        "Rest for 30 minutes",
        "Charge at 0.2C until 3.6 V",
    ]
)

sim = pybamm.Simulation(
    model,
    parameter_values=parameter_values,
    experiment=experiment,
)
sol = sim.solve(calc_esoh=False)

### Plot results

`sol.plot()` does not support blended electrodes, so we plot the key variables directly, including the lithiation of each phase in the blended negative electrode.

In [ ]:
time_h = sol["Time [h]"].entries
voltage = sol["Voltage [V]"].entries
current = -sol["Current [A]"].entries
sto_graphite = sol["Negative electrode primary stoichiometry"].entries
sto_silicon = sol["Negative electrode secondary stoichiometry"].entries

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax_v, ax_i, ax_xli = ax

ax_v.plot(time_h, voltage)
ax_v.set_xlabel("Time [h]")
ax_v.set_ylabel("Voltage [V]")
ax_v.grid()

ax_i.plot(time_h, current)
ax_i.set_xlabel("Time [h]")
ax_i.set_ylabel("Current [A]")
ax_i.grid()

ax_xli.plot(time_h, sto_graphite, label="Graphite (primary)")
ax_xli.plot(time_h, sto_silicon, label="Silicon (secondary)")
ax_xli.set_xlabel("Time [h]")
ax_xli.set_ylabel("Lithiation extent")
ax_xli.legend()
ax_xli.grid()

plt.tight_layout()
plt.show()